In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, default_convert
from torch.nn.utils.rnn import pad_sequence
from typing import List
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import pandas as pd
import os
from sklearn.metrics import confusion_matrix
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR


In [40]:
from models.segment_break_detection.utils.dataset import BreakDataset
from models.segment_break_detection.utils.cnn_rnn import CNNRNN

# AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
# LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
shared_dir = "../../data/clean"
AUDIO_DIR = f"{shared_dir}/audio/vocals"
LABEL_DIR = f"{shared_dir}/segment_breaks"

interval_width = 20  # ms

MODEL_PATH = 'best_model.pth'

In [58]:
class TemporalCNN(nn.Module):
    def __init__(self, input_channels=32):
        super(TemporalCNN, self).__init__()

        # Encoder dimensions
        ls1 = 64
        ls2 = 128
        ls3 = 64

        kys = 5
        ps = 2

        # Encoder path
        self.enc1 = nn.Sequential(
            nn.Conv1d(input_channels, ls1, kernel_size=kys, padding=ps),
            nn.ReLU(),
            nn.BatchNorm1d(ls1)
        )

        self.enc2 = nn.Sequential(
            nn.Conv1d(ls1, ls2, kernel_size=kys, padding=ps),
            nn.ReLU(),
            nn.BatchNorm1d(ls2)
        )

        self.enc3 = nn.Sequential(
            nn.Conv1d(ls2, ls3, kernel_size=kys, padding=ps),
            nn.ReLU(),
            nn.BatchNorm1d(ls3)
        )

        # Global pooling and fully connected layer
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(ls3, 1)

    def forward(self, x, mask=None):
        # x shape: (batch_size, time_steps, features)
        x = x.transpose(1, 2)  # Convert to (batch_size, features, time_steps)

        # Encoder path
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        # Global pooling
        pooled = self.global_pool(e3).squeeze(-1)

        # Fully connected layer
        out = self.fc(pooled)

        # Expand back to time steps if needed
        out = out.repeat(1, x.size(2))

        return out


In [ ]:
def custom_collate(batch, buffer_size=2):
    # Separate file indices, spectrograms and labels
    file_indices, spectrograms, labels = zip(*batch)

    # Apply buffer expansion to labels
    buffered_labels = [torch.tensor(expand_ones(label.numpy(), buffer_size)) for label in labels]

    # Pad spectrograms and buffered labels
    padded_specs = pad_sequence(spectrograms, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(buffered_labels, batch_first=True, padding_value=0)

    # Create mask for padded sequences
    mask = (padded_specs != 0).any(dim=-1).float()

    # Convert file indices to tensor
    file_indices = torch.tensor(file_indices)

    return file_indices, padded_specs, padded_labels, mask

In [48]:
def expand_ones(arr, n):
    arr = np.array(arr)
    length = len(arr)
    expanded = np.zeros_like(arr)

    # Iterate through the array
    for i in range(length):
        if arr[i] == 1:
            # Ensure boundaries are handled
            start = max(0, i)
            end = min(length, i + n + 1)
            expanded[start:end] = 1

    return expanded

In [49]:
dataset = BreakDataset(AUDIO_DIR, LABEL_DIR, interval_width=interval_width, include_index=True)
indices = list(range(len(dataset)))
train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42)

Loading dataset with include_index=True...


In [50]:
train_loader = DataLoader(
    [dataset[i] for i in train_indices],
    batch_size=8,
    shuffle=True,
    collate_fn=lambda batch: custom_collate(batch, buffer_size=5)  # Adjust buffer_size as needed
)
test_loader = DataLoader(
    [dataset[i] for i in test_indices],
    batch_size=8,
    shuffle=False,
    collate_fn=lambda batch: custom_collate(batch, buffer_size=5)  # Same buffer size for consistency
)

In [51]:
# which files are in the test set?
test_files = [dataset[i][0] for i in test_indices]
test_files

[78, 90, 0, 55, 14, 91, 24, 20, 9, 38, 40, 18, 4, 21]

In [24]:
# import copy
# 
# # Training setup
# def train_model(train_loader=train_loader, test_loader=test_loader, num_epochs=30):
#     # Initialize model, loss function, and optimizer
#     device = torch.device('cuda')
#     model = TemporalCNN().to(device)
#     criterion = nn.BCEWithLogitsLoss()
#     optimizer = optim.Adam(model.parameters(), lr=0.01)
#     scheduler = StepLR(optimizer, step_size=5, gamma=0.1)
# 
#     best_val_accuracy = 0.0
#     best_model_state = copy.deepcopy(model.state_dict())
#     patience = 3
#     trigger_times = 0
# 
#     # Training loop
#     for epoch in range(num_epochs):
#         model.train()
#         train_loss = 0
#         for batch_idx, (file_indices, specs, labels, mask) in enumerate(train_loader):
#             specs = specs.to(device).float()
#             labels = labels.to(device).float()
#             mask = mask.to(device)
# 
#             optimizer.zero_grad()
#             outputs = model(specs)
# 
#             # Compute loss with masking
#             loss = criterion(outputs, labels)
#             loss = (loss * mask).mean()
# 
#             loss.backward()
#             optimizer.step()
#             scheduler.step()
# 
#             train_loss += loss.item()
# 
#             if batch_idx % 10 == 0:
#                 print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
# 
#         # Validation phase
#         model.eval()
#         val_loss = 0
#         correct = 0
#         total = 0
# 
#         with torch.no_grad():
#             for file_indices, specs, labels, mask in test_loader:
#                 specs = specs.to(device).float()
#                 labels = labels.to(device).float()
#                 mask = mask.to(device)
# 
#                 outputs = model(specs)
# 
#                 # Compute loss with masking
#                 loss = criterion(outputs, labels)
#                 loss = (loss * mask).mean()
#                 val_loss += loss.item()
# 
#                 # Compute accuracy with masking
#                 predictions = (torch.sigmoid(outputs) > 0.5).float()
#                 correct += ((predictions == labels) * mask).sum().item()
#                 total += mask.sum().item()
# 
#         epoch_val_accuracy = 100 * correct / total
#         print(f'Epoch: {epoch+1}/{num_epochs}')
#         print(f'Training Loss: {train_loss/len(train_loader):.4f}')
#         print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
#         print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
#         print('-' * 60)
# 
#         if epoch_val_accuracy > best_val_accuracy:
#             best_val_accuracy = epoch_val_accuracy
#             best_model_state = copy.deepcopy(model.state_dict())
#             trigger_times = 0
#         else:
#             trigger_times += 1
#             if trigger_times >= patience:
#                 print("Early stopping!")
#                 model.load_state_dict(best_model_state)
#                 return
# 
# if __name__ == "__main__":
#     train_model()


Epoch: 1/30, Batch: 0, Loss: 0.5628
Epoch: 1/30
Training Loss: 0.5628
Validation Loss: 0.4972
Validation Accuracy: 89.95%
------------------------------------------------------------
Epoch: 2/30, Batch: 0, Loss: 0.6996
Epoch: 2/30
Training Loss: 0.6996
Validation Loss: 0.5985
Validation Accuracy: 89.07%
------------------------------------------------------------
Epoch: 3/30, Batch: 0, Loss: 0.5756
Epoch: 3/30
Training Loss: 0.5756
Validation Loss: 0.4389
Validation Accuracy: 89.87%
------------------------------------------------------------
Epoch: 4/30, Batch: 0, Loss: 0.4917
Epoch: 4/30
Training Loss: 0.4917
Validation Loss: 0.3799
Validation Accuracy: 89.95%
------------------------------------------------------------
Early stopping!


In [60]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix
# import os
# import numpy as np
# import copy
# 
# def weighted_binary_cross_entropy(outputs, labels, pos_weight=10.0):
#     """
#     Custom weighted binary cross entropy loss
#     
#     Args:
#     - outputs: model predictions (logits)
#     - labels: true labels
#     - pos_weight: weight given to positive class (default 10)
#     
#     Returns:
#     - weighted loss
#     """
#     # Convert logits to probabilities
#     epsilon = 1e-7
#     outputs = torch.clamp(torch.sigmoid(outputs), epsilon, 1 - epsilon)
# 
#     # Compute individual losses
#     pos_loss = -labels * torch.log(outputs)
#     neg_loss = -(1 - labels) * torch.log(1 - outputs)
# 
#     # Apply weights
#     weighted_pos_loss = pos_loss * pos_weight
#     weighted_neg_loss = neg_loss
# 
#     # Combine losses
#     return weighted_pos_loss + weighted_neg_loss
# 
# def train_model(train_loader=train_loader, test_loader=test_loader, num_epochs=30, output_dir='prediction_outputs'):
#     # Create output directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
# 
#     # Initialize model, loss function, and optimizer
#     device = torch.device('cuda')
#     model = TemporalCNN().to(device)
# 
#     # Use the custom weighted loss function
#     criterion = lambda outputs, labels: weighted_binary_cross_entropy(outputs, labels, pos_weight=10.0)
# 
#     optimizer = optim.Adam(model.parameters(), lr=0.01)
#     scheduler = StepLR(optimizer, step_size=5, gamma=0.1)
# 
#     # Rest of the training loop remains the same
#     best_val_accuracy = 0.0
#     best_model_state = copy.deepcopy(model.state_dict())
#     patience = 3
#     trigger_times = 0
# 
#     for epoch in range(num_epochs):
#         model.train()
#         train_loss = 0
#         for batch_idx, (file_indices, specs, labels, mask) in enumerate(train_loader):
#             specs = specs.to(device).float()
#             labels = labels.to(device).float()
#             mask = mask.to(device)
# 
#             optimizer.zero_grad()
#             outputs = model(specs)
# 
#             # Compute loss with masking and weighting
#             loss = criterion(outputs, labels)
#             loss = (loss * mask).mean()
# 
#             loss.backward()
#             optimizer.step()
#             scheduler.step()
# 
#             train_loss += loss.item()
# 
#             if batch_idx % 10 == 0:
#                 print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
# 
# 
#         # Validation phase
#         model.eval()
#         val_loss = 0
#         correct = 0
#         total = 0
# 
#         # Lists to store all predictions and labels
#         all_predictions = []
#         all_labels = []
#         all_file_indices = []
# 
#         with torch.no_grad():
#             for file_indices, specs, labels, mask in test_loader:
#                 specs = specs.to(device).float()
#                 labels = labels.to(device).float()
#                 mask = mask.to(device)
# 
#                 outputs = model(specs)
#                 predictions = (torch.sigmoid(outputs) > 0.5).float()
# 
#                 # Store predictions and labels for each file
#                 for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
#                     # Only consider non-padded timesteps
#                     valid_pred = pred[m.bool()].cpu().numpy()
#                     valid_label = label[m.bool()].cpu().numpy()
# 
#                     # Save to text file
#                     with open(os.path.join(output_dir, f'{idx.item()}.txt'), 'w') as f:
#                         np.savetxt(f, [valid_pred, valid_label], fmt='%d')
# 
#                     # Append to lists for confusion matrix
#                     all_predictions.extend(valid_pred)
#                     all_labels.extend(valid_label)
#                     all_file_indices.extend([idx.item()] * len(valid_pred))
# 
#                 # Compute loss and accuracy with masking
#                 loss = criterion(outputs, labels)
#                 loss = (loss * mask).mean()
#                 val_loss += loss.item()
#                 correct += ((predictions == labels) * mask).sum().item()
#                 total += mask.sum().item()
# 
#         # Create and save confusion matrix
#         cm = confusion_matrix(all_labels, all_predictions)
#         plt.figure(figsize=(8, 6))
#         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
#         plt.title(f'Confusion Matrix - Epoch {epoch+1}')
#         plt.ylabel('True Label')
#         plt.xlabel('Predicted Label')
#         plt.savefig(os.path.join(output_dir, f'confusion_matrix_epoch_{epoch+1}.png'))
#         plt.close()
# 
#         epoch_val_accuracy = 100 * correct / total
#         print(f'Epoch: {epoch+1}/{num_epochs}')
#         print(f'Training Loss: {train_loss/len(train_loader):.4f}')
#         print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
#         print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
#         print('-' * 60)
# 
#         # Early stopping logic
#         if epoch_val_accuracy > best_val_accuracy:
#             best_val_accuracy = epoch_val_accuracy
#             best_model_state = copy.deepcopy(model.state_dict())
#             trigger_times = 0
#         else:
#             trigger_times += 1
#             if trigger_times >= patience:
#                 print("Early stopping!")
#                 model.load_state_dict(best_model_state)
# 
#                 # Generate final confusion matrix and predictions with best model
#                 model.eval()
#                 final_predictions = []
#                 final_labels = []
# 
#                 with torch.no_grad():
#                     for file_indices, specs, labels, mask in test_loader:
#                         specs = specs.to(device).float()
#                         labels = labels.to(device).float()
#                         mask = mask.to(device)
# 
#                         outputs = model(specs)
#                         predictions = (torch.sigmoid(outputs) > 0.5).float()
# 
#                         for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
#                             valid_pred = pred[m.bool()].cpu().numpy()
#                             valid_label = label[m.bool()].cpu().numpy()
# 
#                             # Save final predictions
#                             with open(os.path.join(output_dir, f'{idx.item()}_final.txt'), 'w') as f:
#                                 np.savetxt(f, [valid_pred, valid_label], fmt='%d')
# 
#                             final_predictions.extend(valid_pred)
#                             final_labels.extend(valid_label)
# 
#                 # Create final confusion matrix
#                 final_cm = confusion_matrix(final_labels, final_predictions)
#                 plt.figure(figsize=(8, 6))
#                 sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues')
#                 plt.title('Final Confusion Matrix (Best Model)')
#                 plt.ylabel('True Label')
#                 plt.xlabel('Predicted Label')
#                 plt.savefig(os.path.join(output_dir, 'final_confusion_matrix.png'))
#                 plt.close()
# 
#                 return
# 
# if __name__ == "__main__":
#     train_model()


Epoch: 1/30, Batch: 0, Loss: 1.9641
Epoch: 1/30
Training Loss: 2.1579
Validation Loss: 1.3554
Validation Accuracy: 30.10%
------------------------------------------------------------
Epoch: 2/30, Batch: 0, Loss: 1.8969
Epoch: 2/30
Training Loss: 1.9531
Validation Loss: 1.5023
Validation Accuracy: 30.10%
------------------------------------------------------------
Epoch: 3/30, Batch: 0, Loss: 1.7179
Epoch: 3/30
Training Loss: 1.9415
Validation Loss: 1.6178
Validation Accuracy: 30.10%
------------------------------------------------------------
Epoch: 4/30, Batch: 0, Loss: 1.7019
Epoch: 4/30
Training Loss: 1.9022
Validation Loss: 1.6497
Validation Accuracy: 30.10%
------------------------------------------------------------
Early stopping!


In [73]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix
# import os
# import numpy as np
# import copy
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.optim.lr_scheduler import StepLR
# 
# class DynamicLossWeighter:
#     def __init__(self, initial_false_positive_weight=1.0):
#         """
#         Dynamic loss weighting system focused on false positive penalty
#         
#         Args:
#         - initial_false_positive_weight: Initial weight for false positive penalty
#         """
#         self.false_positive_weight = initial_false_positive_weight
# 
#         # Tracking mistake history
#         self.mistake_history = {
#             'false_positives': [],
#             'false_negatives': [],
#             'true_positives': [],
#             'true_negatives': []
#         }
# 
#         # Adaptive parameters
#         self.max_weight_limit = 50.0
#         self.min_weight_limit = 1.0
# 
#     def compute_false_positive_weighted_loss(self, labels, predictions):
#         """
#         Compute a loss weight that heavily penalizes false positives
#         
#         Args:
#         - labels: true labels
#         - predictions: model predictions
#         
#         Returns:
#         - Updated false positive weight
#         """
#         # Compute confusion matrix
#         tn, fp, fn, tp = confusion_matrix(labels, predictions).ravel()
# 
#         # Calculate false positive rate
#         false_positive_rate = fp / (fp + tn + 1e-7)
# 
#         # Compute weights
#         # Exponential increase in penalty for false positives
#         false_positive_penalty = np.exp(false_positive_rate * 9)  # Exponential scaling
# 
#         # Limit the penalty to prevent extreme values
#         self.false_positive_weight = max(
#             self.min_weight_limit,
#             min(self.max_weight_limit, false_positive_penalty)
#         )
# 
#         # Store current epoch's mistake statistics
#         self.mistake_history['false_positives'].append(fp)
#         self.mistake_history['false_negatives'].append(fn)
#         self.mistake_history['true_positives'].append(tp)
#         self.mistake_history['true_negatives'].append(tn)
# 
#         # Print detailed performance breakdown
#         print(f"\nConfusion Matrix Breakdown:")
#         print(f"True Negatives: {tn}, False Positives: {fp}")
#         print(f"True Positives: {tp}, False Negatives: {fn}")
#         print(f"False Positive Rate: {false_positive_rate:.4f}")
#         print(f"False Positive Penalty Weight: {self.false_positive_weight:.2f}")
# 
#         return self.false_positive_weight
# 
# def weighted_binary_cross_entropy(outputs, labels, false_positive_weight=1.0):
#     """
#     Custom weighted binary cross entropy loss with false positive penalty
#     
#     Args:
#     - outputs: model predictions (logits)
#     - labels: true labels
#     - false_positive_weight: weight to penalize false positives
#     
#     Returns:
#     - weighted loss
#     """
#     # Convert logits to probabilities
#     epsilon = 1e-7
#     outputs = torch.clamp(torch.sigmoid(outputs), epsilon, 1 - epsilon)
# 
#     # Compute individual losses
#     # Positive class loss (true positives and false negatives)
#     pos_loss = -labels * torch.log(outputs)
# 
#     # Negative class loss (true negatives and false positives)
#     # Apply additional weight to false positives
#     neg_loss = -(1 - labels) * torch.log(1 - outputs)
#     false_positive_mask = (1 - labels)  # Mask for potential false positives
# 
#     # Apply false positive penalty
#     weighted_neg_loss = neg_loss * (1 + false_positive_mask * (false_positive_weight - 1))
# 
#     # Combine losses
#     return pos_loss + weighted_neg_loss
# 
# def train_model(train_loader, test_loader, num_epochs=30, output_dir='prediction_outputs'):
#     # Create output directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
# 
#     # Initialize model, loss weighter, and optimizer
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = TemporalCNN().to(device)
# 
#     # Initialize dynamic loss weighter
#     loss_weighter = DynamicLossWeighter()
# 
#     # Optimizer and scheduler
#     optimizer = optim.Adam(model.parameters(), lr=0.05)
#     scheduler = StepLR(optimizer, step_size=5, gamma=0.1)
# 
#     # Early stopping parameters
#     best_val_accuracy = 0.0
#     best_model_state = copy.deepcopy(model.state_dict())
#     patience = 10
#     trigger_times = 0
# 
#     # Training loop
#     for epoch in range(num_epochs):
#         model.train()
#         train_loss = 0
# 
#         # Training phase
#         for batch_idx, (file_indices, specs, labels, mask) in enumerate(train_loader):
#             specs = specs.to(device).float()
#             labels = labels.to(device).float()
#             mask = mask.to(device)
# 
#             optimizer.zero_grad()
#             outputs = model(specs)
# 
#             # Dynamic loss function with false positive penalty
#             current_false_positive_weight = loss_weighter.false_positive_weight
#             criterion = lambda outputs, labels: weighted_binary_cross_entropy(
#                 outputs, labels,
#                 false_positive_weight=current_false_positive_weight
#             )
# 
#             # Compute loss with masking and weighting
#             loss = criterion(outputs, labels)
#             loss = (loss * mask).mean()
# 
#             loss.backward()
#             optimizer.step()
#             scheduler.step()
# 
#             train_loss += loss.item()
# 
#             if batch_idx % 10 == 0:
#                 print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
# 
#         # Validation phase
#         model.eval()
#         val_loss = 0
#         correct = 0
#         total = 0
# 
#         # Lists to store all predictions and labels
#         all_predictions = []
#         all_labels = []
#         all_file_indices = []
# 
#         with torch.no_grad():
#             for file_indices, specs, labels, mask in test_loader:
#                 specs = specs.to(device).float()
#                 labels = labels.to(device).float()
#                 mask = mask.to(device)
# 
#                 outputs = model(specs)
#                 predictions = (torch.sigmoid(outputs) > 0.5).float()
# 
#                 # Store predictions and labels for each file
#                 for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
#                     # Only consider non-padded timesteps
#                     valid_pred = pred[m.bool()].cpu().numpy()
#                     valid_label = label[m.bool()].cpu().numpy()
# 
#                     # Append to lists for confusion matrix
#                     all_predictions.extend(valid_pred)
#                     all_labels.extend(valid_label)
#                     all_file_indices.extend([idx.item()] * len(valid_pred))
# 
#                 # Compute loss and accuracy with masking
#                 loss = criterion(outputs, labels)
#                 loss = (loss * mask).mean()
#                 val_loss += loss.item()
#                 correct += ((predictions == labels) * mask).sum().item()
#                 total += mask.sum().item()
# 
#         # Dynamically compute false positive weight
#         current_false_positive_weight = loss_weighter.compute_false_positive_weighted_loss(
#             all_labels, all_predictions
#         )
# 
#         # Create and save confusion matrix
#         cm = confusion_matrix(all_labels, all_predictions)
#         plt.figure(figsize=(8, 6))
#         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
#         plt.title(f'Confusion Matrix - Epoch {epoch+1}')
#         plt.ylabel('True Label')
#         plt.xlabel('Predicted Label')
#         plt.savefig(os.path.join(output_dir, f'confusion_matrix_epoch_{epoch+1}.png'))
#         plt.close()
# 
#         # Compute and print epoch metrics
#         epoch_val_accuracy = 100 * correct / total
#         print(f'Epoch: {epoch+1}/{num_epochs}')
#         print(f'Training Loss: {train_loss/len(train_loader):.4f}')
#         print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
#         print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
#         print('-' * 60)
# 
#         # Early stopping logic
#         continue
#         if epoch_val_accuracy > best_val_accuracy:
#             best_val_accuracy = epoch_val_accuracy
#             best_model_state = copy.deepcopy(model.state_dict())
#             trigger_times = 0
#         else:
#             trigger_times += 1
#             if trigger_times >= patience:
#                 print("Early stopping!")
#                 model.load_state_dict(best_model_state)
# 
#                 # Generate final confusion matrix and predictions with best model
#                 model.eval()
#                 final_predictions = []
#                 final_labels = []
# 
#                 with torch.no_grad():
#                     for file_indices, specs, labels, mask in test_loader:
#                         specs = specs.to(device).float()
#                         labels = labels.to(device).float()
#                         mask = mask.to(device)
# 
#                         outputs = model(specs)
#                         predictions = (torch.sigmoid(outputs) > 0.5).float()
# 
#                         for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
#                             valid_pred = pred[m.bool()].cpu().numpy()
#                             valid_label = label[m.bool()].cpu().numpy()
# 
#                             # Save final predictions
#                             with open(os.path.join(output_dir, f'{idx.item()}_final.txt'), 'w') as f:
#                                 np.savetxt(f, [valid_pred, valid_label], fmt='%d')
# 
#                             final_predictions.extend(valid_pred)
#                             final_labels.extend(valid_label)
# 
#                 # Create final confusion matrix
#                 final_cm = confusion_matrix(final_labels, final_predictions)
#                 plt.figure(figsize=(8, 6))
#                 sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues')
#                 plt.title('Final Confusion Matrix (Best Model)')
#                 plt.ylabel('True Label')
#                 plt.xlabel('Predicted Label')
#                 plt.savefig(os.path.join(output_dir, 'final_confusion_matrix.png'))
#                 plt.close()
# 
#                 return model
# 
#     return model
# 
# if __name__ == "__main__":
#     train_model(train_loader, test_loader)


Epoch: 1/30, Batch: 0, Loss: 0.5927

Confusion Matrix Breakdown:
True Negatives: 0, False Positives: 96975
True Positives: 41763, False Negatives: 0
False Positive Rate: 1.0000
False Positive Penalty Weight: 50.00
Epoch: 1/30
Training Loss: 0.5190
Validation Loss: 2.1761
Validation Accuracy: 30.10%
------------------------------------------------------------
Epoch: 2/30, Batch: 0, Loss: 10.5530

Confusion Matrix Breakdown:
True Negatives: 96975, False Positives: 0
True Positives: 0, False Negatives: 41763
False Positive Rate: 0.0000
False Positive Penalty Weight: 1.00
Epoch: 2/30
Training Loss: 8.4072
Validation Loss: 6.2447
Validation Accuracy: 69.90%
------------------------------------------------------------
Epoch: 3/30, Batch: 0, Loss: 0.4767

Confusion Matrix Breakdown:
True Negatives: 96975, False Positives: 0
True Positives: 0, False Negatives: 41763
False Positive Rate: 0.0000
False Positive Penalty Weight: 1.00
Epoch: 3/30
Training Loss: 0.4943
Validation Loss: 0.4468
Validat

In [77]:
!pip install fastdtw

  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-py3-none-any.whl size=3565 sha256=adea34ac8877492ba4176c08d7f94379d2a7164f7e80230d2e7022960c855b78
  Stored in directory: /home/eizigi/.cache/pip/wheels/73/c8/f7/c25448dab74c3acf4848bc25d513c736bb93910277e1528ef4
Successfully built fastdtw


In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import copy
from fastdtw import fastdtw
import numpy as np

def dtw_loss(outputs, targets, mask):
    batch_size = outputs.shape[0]
    total_loss = 0

    # Move tensors to CPU and convert to numpy for fastdtw
    outputs_np = outputs.detach().cpu().numpy()
    targets_np = targets.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()

    for i in range(batch_size):
        # Only consider the timesteps where mask is 1
        valid_indices = np.where(mask_np[i] == 1)[0]
        if len(valid_indices) == 0:
            continue

        output_seq = outputs_np[i, valid_indices]
        target_seq = targets_np[i, valid_indices]

        distance, _ = fastdtw(output_seq, target_seq)
        total_loss += distance

    return torch.tensor(total_loss / batch_size, requires_grad=True, device=outputs.device)

def train_model(train_loader=train_loader, test_loader=test_loader, num_epochs=30):
    # Initialize model and optimizer
    device = torch.device('cuda')
    model = TemporalCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

    best_val_accuracy = 0.0
    best_model_state = copy.deepcopy(model.state_dict())
    patience = 3
    trigger_times = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch_idx, (file_indices, specs, labels, mask) in enumerate(train_loader):
            specs = specs.to(device).float()
            labels = labels.to(device).float()
            mask = mask.to(device)

            optimizer.zero_grad()
            outputs = model(specs)

            # Compute DTW loss
            loss = dtw_loss(outputs, labels, mask)

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')

        # Validation phase
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for file_indices, specs, labels, mask in test_loader:
                specs = specs.to(device).float()
                labels = labels.to(device).float()
                mask = mask.to(device)

                outputs = model(specs)

                # Compute DTW loss for validation
                loss = dtw_loss(outputs, labels, mask)
                val_loss += loss.item()

                # Compute accuracy with masking
                predictions = (torch.sigmoid(outputs) > 0.5).float()
                correct += ((predictions == labels) * mask).sum().item()
                total += mask.sum().item()

        epoch_val_accuracy = 100 * correct / total
        print(f'Epoch: {epoch+1}/{num_epochs}')
        print(f'Training Loss: {train_loss/len(train_loader):.4f}')
        print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
        print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
        print('-' * 60)

        if epoch_val_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_val_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping!")
                model.load_state_dict(best_model_state)
                return

if __name__ == "__main__":
    train_model()


Epoch: 1/30, Batch: 0, Loss: 3538.0528
Epoch: 1/30
Training Loss: 3686.1948
Validation Loss: 3322.6171
Validation Accuracy: 69.90%
------------------------------------------------------------
Epoch: 2/30, Batch: 0, Loss: 3875.3656
Epoch: 2/30
Training Loss: 3653.7057
Validation Loss: 4278.7878
Validation Accuracy: 69.90%
------------------------------------------------------------
Epoch: 3/30, Batch: 0, Loss: 4462.3763
Epoch: 3/30
Training Loss: 3671.5911
Validation Loss: 4761.0499
Validation Accuracy: 69.90%
------------------------------------------------------------
Epoch: 4/30, Batch: 0, Loss: 3709.1618
Epoch: 4/30
Training Loss: 3639.9710
Validation Loss: 4809.4794
Validation Accuracy: 69.90%
------------------------------------------------------------
Early stopping!


In [79]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import numpy as np
import copy

def weighted_binary_cross_entropy(outputs, labels, pos_weight=10.0):
    """
    Custom weighted binary cross entropy loss

    Args:
    - outputs: model predictions (logits)
    - labels: true labels
    - pos_weight: weight given to positive class (default 10)

    Returns:
    - weighted loss
    """
    # Convert logits to probabilities
    epsilon = 1e-7
    outputs = torch.clamp(torch.sigmoid(outputs), epsilon, 1 - epsilon)

    # Compute individual losses
    pos_loss = -labels * torch.log(outputs)
    neg_loss = -(1 - labels) * torch.log(1 - outputs)

    # Apply weights
    weighted_pos_loss = pos_loss * pos_weight
    weighted_neg_loss = neg_loss

    # Combine losses
    return weighted_pos_loss + weighted_neg_loss

def train_model(train_loader=train_loader, test_loader=test_loader, num_epochs=30, output_dir='prediction_outputs'):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Initialize model, loss function, and optimizer
    device = torch.device('cuda')
    model = TemporalCNN().to(device)

    # Use the custom weighted loss function
    criterion = lambda outputs, labels: weighted_binary_cross_entropy(outputs, labels, pos_weight=10.0)

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

    # Rest of the training loop remains the same
    best_val_accuracy = 0.0
    best_model_state = copy.deepcopy(model.state_dict())
    patience = 3
    trigger_times = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch_idx, (file_indices, specs, labels, mask) in enumerate(train_loader):
            specs = specs.to(device).float()
            labels = labels.to(device).float()
            mask = mask.to(device)

            optimizer.zero_grad()
            outputs = model(specs)

            # Compute loss with masking and weighting
            loss = criterion(outputs, labels)
            loss = (loss * mask).mean()

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')


        # Validation phase
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        # Lists to store all predictions and labels
        all_predictions = []
        all_labels = []
        all_file_indices = []

        with torch.no_grad():
            for file_indices, specs, labels, mask in test_loader:
                specs = specs.to(device).float()
                labels = labels.to(device).float()
                mask = mask.to(device)

                outputs = model(specs)
                predictions = (torch.sigmoid(outputs) > 0.5).float()

                # Store predictions and labels for each file
                for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
                    # Only consider non-padded timesteps
                    valid_pred = pred[m.bool()].cpu().numpy()
                    valid_label = label[m.bool()].cpu().numpy()

                    # Save to text file
                    with open(os.path.join(output_dir, f'{idx.item()}.txt'), 'w') as f:
                        np.savetxt(f, [valid_pred, valid_label], fmt='%d')

                    # Append to lists for confusion matrix
                    all_predictions.extend(valid_pred)
                    all_labels.extend(valid_label)
                    all_file_indices.extend([idx.item()] * len(valid_pred))

                # Compute loss and accuracy with masking
                loss = criterion(outputs, labels)
                loss = (loss * mask).mean()
                val_loss += loss.item()
                correct += ((predictions == labels) * mask).sum().item()
                total += mask.sum().item()

        # Create and save confusion matrix
        cm = confusion_matrix(all_labels, all_predictions)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - Epoch {epoch+1}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.savefig(os.path.join(output_dir, f'confusion_matrix_epoch_{epoch+1}.png'))
        plt.close()

        epoch_val_accuracy = 100 * correct / total
        print(f'Epoch: {epoch+1}/{num_epochs}')
        print(f'Training Loss: {train_loss/len(train_loader):.4f}')
        print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
        print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
        print('-' * 60)

        # Early stopping logic
        if epoch_val_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_val_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping!")
                model.load_state_dict(best_model_state)

                # Generate final confusion matrix and predictions with best model
                model.eval()
                final_predictions = []
                final_labels = []

                with torch.no_grad():
                    for file_indices, specs, labels, mask in test_loader:
                        specs = specs.to(device).float()
                        labels = labels.to(device).float()
                        mask = mask.to(device)

                        outputs = model(specs)
                        predictions = (torch.sigmoid(outputs) > 0.5).float()

                        for idx, pred, label, m in zip(file_indices, predictions, labels, mask):
                            valid_pred = pred[m.bool()].cpu().numpy()
                            valid_label = label[m.bool()].cpu().numpy()

                            # Save final predictions
                            with open(os.path.join(output_dir, f'{idx.item()}_final.txt'), 'w') as f:
                                np.savetxt(f, [valid_pred, valid_label], fmt='%d')

                            final_predictions.extend(valid_pred)
                            final_labels.extend(valid_label)

                # Create final confusion matrix
                final_cm = confusion_matrix(final_labels, final_predictions)
                plt.figure(figsize=(8, 6))
                sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues')
                plt.title('Final Confusion Matrix (Best Model)')
                plt.ylabel('True Label')
                plt.xlabel('Predicted Label')
                plt.savefig(os.path.join(output_dir, 'final_confusion_matrix.png'))
                plt.close()

                return

if __name__ == "__main__":
    train_model()


IndexError: Dimension out of range (expected to be in range of [-2, 1], but got 2)